# Handle Cube Files

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# AiiDA imports.
%load_ext aiida
%aiida

import urllib

import traitlets as tl
from aiida import orm

from surfaces_tools.widgets import cube  # import DisplayRenderedCubes, HandleCubeFiles

In [ ]:
cube_files = cube.HandleCubeFiles()


def load_cube_page(node_pk):
    node = orm.load_node(node_pk)
    caller = node.caller
    if caller is not None:
        if caller.label == "CP2K_ORBITALS":
            parameters = caller.outputs.dft_output_parameters
        elif caller.label == "CP2K_GeoOpt":
            parameters = caller.outputs.output_parameters
        else:
            parameters = None
        if parameters is not None:
            cube_files.uks = parameters["dft_type"] == "UKS"
            cube_files.nel_up = int(parameters["init_nel_spin1"])
            cube_files.nel_dw = int(parameters["init_nel_spin2"])
    cube_files.node_pk = node_pk


url = urllib.parse.urlsplit(jupyter_notebook_url)
try:
    load_cube_page(int(urllib.parse.parse_qs(url.query)["pk"][0]))
except KeyError:
    pass
except Exception:
    cube_files.node_pk = None
    cube_files.error_message.value = (
        "Unable to load this cube result. Select a completed cubehelper."
    )
display(cube_files)

In [ ]:
display_images = cube.DisplayRenderedCubes()
tl.dlink((cube_files, "remote_data_uuid"), (display_images, "remote_data_uuid"))
display(display_images)